In [ ]:
# Script that analyses area fold changes in transcribing and non-transcribing NCs shown in Figure 5

# Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel

# Load dataset
df = pd.read_csv('AreasInWindowsNaNMean.csv')

# Identify window columns
windows = [col for col in df.columns if col != 'Signalling']

# Calculate fold change
df_fc = df.copy()
df_fc[windows] = df_fc[windows].div(df_fc[windows[0]], axis=0)

# Split signalling vs non-signalling
df_signal = df_fc[df_fc['Signalling'] == 1].drop(columns=['Signalling'])
df_nosignal = df_fc[df_fc['Signalling'] == 0].drop(columns=['Signalling'])

# Heatmap plotting function
def plot_heatmap(df, title, outfile):
    plt.figure(figsize=(6, 6))
    ax = sns.heatmap(
        df,
        cmap="GnBu",
        cbar=True,
        vmin=0.5, vmax=2.0,
        linewidths=0,
        linecolor="none",
        square=False
    )

    # Rasterize heatmap only (avoids grid artifacts in PDF)
    ax.collections[0].set_rasterized(True)

    plt.title(title)
    plt.xlabel("Windows")
    plt.ylabel("Cells")
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()

# Plot heatmaps
plot_heatmap(
    df_signal,
    "Fold-change areas across windows (Signalling cells)",
    "areas_fc_heatmap_signalling_NaNMean.pdf"
)

plot_heatmap(
    df_nosignal,
    "Fold-change areas across windows (Non-signalling cells)",
    "areas_fc_heatmap_nonsignalling_NaNMean.pdf"
)

# Stats + paired t-tests
def window_stats_and_tests(df_fc, label):
    print(f"\n=== {label} cells ===")

    windows = df_fc.columns.tolist()

    # Mean ± SD per window
    stats = df_fc.agg(['mean', 'std']).T
    print("\nMean ± SD fold change per window:")
    print(stats)

    # Paired t-tests between successive windows
    print("\nPaired t-tests (successive windows):")
    for w1, w2 in zip(windows[:-1], windows[1:]):
        tstat, pval = ttest_rel(df_fc[w1], df_fc[w2], nan_policy='omit')
        print(f"{w1} → {w2}: t = {tstat:.3f}, p = {pval:.4e}")

    return stats

# Convert to long format
def to_long(df_fc):
    df_long = df_fc.copy()
    df_long['CellID'] = df_long.index
    df_long = df_long.melt(
        id_vars='CellID',
        var_name='Window',
        value_name='FoldChange'
    )
    return df_long

# Boxplot + datapoints + paired lines
def plot_boxplot_with_lines(df_long, title, outfile):
    plt.figure(figsize=(7, 5))

    sns.boxplot(
        data=df_long,
        x='Window',
        y='FoldChange',
        color='lightgray',
        showfliers=False
    )

    sns.stripplot(
        data=df_long,
        x='Window',
        y='FoldChange',
        color='black',
        size=4,
        jitter=0.15,
        alpha=0.7
    )

    # Paired lines
    for cell_id, d in df_long.groupby('CellID'):
        plt.plot(
            d['Window'],
            d['FoldChange'],
            color='black',
            alpha=0.3,
            linewidth=0.7
        )
    plt.ylim(0.2,2.5)
    plt.axhline(1, color='red', linestyle='--', linewidth=1)
    plt.ylabel("Apical area fold change")
    plt.xlabel("Time window")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()

# Run stats + boxplots (signalling)
stats_signal = window_stats_and_tests(df_signal, "Signalling")
df_signal_long = to_long(df_signal)

plot_boxplot_with_lines(
    df_signal_long,
    "Apical area fold change (Signalling cells)",
    "areas_fc_boxplot_signalling_NaNMean.pdf"
)

# Run stats + boxplots (non-signalling)
stats_nosignal = window_stats_and_tests(df_nosignal, "Non-signalling")
df_nosignal_long = to_long(df_nosignal)

plot_boxplot_with_lines(
    df_nosignal_long,
    "Apical area fold change (Non-signalling cells)",
    "areas_fc_boxplot_nonsignalling_NaNMean.pdf"
)
df_fc.to_csv("areas_foldchange_dataset_used.csv", index=False)
